In [4]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from app import create_app
from models import db, Symptom, Condition
from symptom_matcher import match_symptoms, get_severity_label

app = create_app()
print("Ready ✓")

Ready ✓


In [5]:
with app.app_context():
    # Simulate a user reporting fever, chills, body aches
    test_answers = {
        'q_1': 'Fever,Chills,Body Aches,Headache,Fatigue',
        'q_2': '1–3 days',
        'q_3': 'Moderate — I am struggling with some activities',
        'q_4': 'None of the above'
    }

    results = match_symptoms(test_answers)

    for r in results:
        info = get_severity_label(r['severity'])
        print(f"{info['icon']} {r['condition'].name}")
        print(f"   Base score:  {r['base_score']}")
        print(f"   Final score: {r['final_score']}")
        print(f"   Matched:     {', '.join(r['matched'])}\n")

In [7]:
import pandas as pd

with app.app_context():
    test_answers = {'q_1': 'Fever,Chills,Body Aches,Headache,Fatigue,Nausea'}
    results = match_symptoms(test_answers)

    df = pd.DataFrame([{
        'Condition':   r['condition'].name,
        'Severity':    r['severity'],
        'Base Score':  r['base_score'],
        'Final Score': r['final_score'],
        'Matched':     len(r['matched'])
    } for r in results])

    df

ModuleNotFoundError: No module named 'pandas'

In [8]:
import matplotlib.pyplot as plt

if not df.empty:
    df.plot.barh(
        x='Condition',
        y='Final Score',
        color='#1A7A7A',
        legend=False,
        figsize=(8, 4)
    )
    plt.title('Condition Match Scores')
    plt.xlabel('Final Score')
    plt.tight_layout()
    plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [9]:
from api_client import get_drug_warnings

with app.app_context():
    results = match_symptoms({'q_1': 'Fever,Chills,Body Aches'})
    for r in results[:3]:
        drugs = get_drug_warnings(r['condition'].name)
        print(f"\n{r['condition'].name}")
        for d in drugs:
            print(f"  → {d['brand_name']}: {d['warnings'][:100]}...")